In [3]:
#Upload the dataset file
from google.colab import files

uploaded = files.upload()

Saving survey.csv to survey (1).csv


In [4]:
import pandas as pd
import numpy as np

# Load the original dataset
df = pd.read_csv("survey.csv", encoding="latin1")

# Make a copy for preprocessing
#기존 데이터를 유지하기 위해서 copy사용
df_clean = df.copy()

In [5]:
#Check basic information of the dataset
#긴단하게 기존 데이터셋 행/열 개수, column 이름, 각 column별 missing values 개수 파악

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns)

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (1259, 27)

Column names:
Index(['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed',
       'family_history', 'treatment', 'work_interfere', 'no_employees',
       'remote_work', 'tech_company', 'benefits', 'care_options',
       'wellness_program', 'seek_help', 'anonymity', 'leave',
       'mental_health_consequence', 'phys_health_consequence', 'coworkers',
       'supervisor', 'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence', 'comments'],
      dtype='object')

Missing values:
Timestamp                       0
Age                             0
Gender                          0
Country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                    

In [6]:
#Drop unnecessary columns
#필요없는 Column 제거하기(Timestamp, comments, state) /  27 -> 24개로 변경

# Columns to remove
drop_columns = ["Timestamp", "comments", "state"]

# Drop columns from the copied dataset
df_clean = df_clean.drop(columns=drop_columns)

# Check result
print("Before dropping columns:", df.shape)
print("After dropping columns:", df_clean.shape)

# Check remaining columns
df_clean.columns

Before dropping columns: (1259, 27)
After dropping columns: (1259, 24)


Index(['Age', 'Gender', 'Country', 'self_employed', 'family_history',
       'treatment', 'work_interfere', 'no_employees', 'remote_work',
       'tech_company', 'benefits', 'care_options', 'wellness_program',
       'seek_help', 'anonymity', 'leave', 'mental_health_consequence',
       'phys_health_consequence', 'coworkers', 'supervisor',
       'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence'],
      dtype='object')

#Dirty data 처리 중 Age 처리
### {Age < 18 | Age > 80}인 값 찾기 -> 비정상 값으로 판단 -> Nan으로 변경 -> Median 값으로 대체


In [7]:
#Handle dirty data in Age

# Check abnormal Age values before cleaning
print("Age values before cleaning:")
print(df_clean["Age"].describe())

# Find abnormal age values
invalid_age = (df_clean["Age"] < 18) | (df_clean["Age"] > 80)

print("\nNumber of invalid Age values:", invalid_age.sum())
print("\nInvalid Age values:")
print(df_clean.loc[invalid_age, "Age"])

# Replace invalid Age values with NaN
df_clean.loc[invalid_age, "Age"] = np.nan

# Replace missing Age values with median
age_median = df_clean["Age"].median()
df_clean["Age"] = df_clean["Age"].fillna(age_median)

print("\nAge median used for replacement:", age_median)
print("\nMissing values in Age after cleaning:", df_clean["Age"].isnull().sum())

print("\nAge values after cleaning:")
print(df_clean["Age"].describe())

Age values before cleaning:
count    1.259000e+03
mean     7.942815e+07
std      2.818299e+09
min     -1.726000e+03
25%      2.700000e+01
50%      3.100000e+01
75%      3.600000e+01
max      1.000000e+11
Name: Age, dtype: float64

Number of invalid Age values: 8

Invalid Age values:
143             -29
364             329
390     99999999999
715           -1726
734               5
989               8
1090             11
1127             -1
Name: Age, dtype: int64

Age median used for replacement: 31.0

Missing values in Age after cleaning: 0

Age values after cleaning:
count    1259.000000
mean       32.069897
std         7.265565
min        18.000000
25%        27.000000
50%        31.000000
75%        36.000000
max        72.000000
Name: Age, dtype: float64


#Gender 값 통일
### 어떤 값이 있었는지 확인하고 값 변경하기

In [8]:
# Clean dirty data in Gender

# Check Gender values before cleaning
print("Gender values before cleaning:")
print(df_clean["Gender"].value_counts())

# Function to clean Gender values
def clean_gender(gender):
    gender = str(gender).strip().lower()

    male_values = [
        "male", "m", "male-ish", "maile", "mal", "male (cis)",
        "make", "male ", "man", "msle", "mail", "cis male",
        "cis man", "ostensibly male, unsure what that really means"
    ]

    female_values = [
        "female", "f", "woman", "female ", "femake",
        "female (cis)", "cis female", "femail"
    ]

    if gender in male_values:
        return "Male"
    elif gender in female_values:
        return "Female"
    else:
        return "Other"

# Apply the function
df_clean["Gender"] = df_clean["Gender"].apply(clean_gender)

# Check Gender values after cleaning
print("\nGender values after cleaning:")
print(df_clean["Gender"].value_counts())

Gender values before cleaning:
Gender
Male                                              615
male                                              206
Female                                            121
M                                                 116
female                                             62
F                                                  38
m                                                  34
f                                                  15
Make                                                4
Male                                                3
Woman                                               3
Cis Male                                            2
Man                                                 2
Female                                              2
Female (trans)                                      2
Male-ish                                            1
Trans-female                                        1
Male (CIS)                                  

#Missung values 처리
### self_employed와 work_interfere의 빈 값을 mode로 대체하기


In [9]:
# Handle missing values

# Check missing values before handling
print("Missing values before handling:")
print(df_clean[["self_employed", "work_interfere", "Age"]].isnull().sum())

# Replace missing values in self_employed with mode
self_employed_mode = df_clean["self_employed"].mode()[0]
df_clean["self_employed"] = df_clean["self_employed"].fillna(self_employed_mode)

# Replace missing values in work_interfere with mode
work_interfere_mode = df_clean["work_interfere"].mode()[0]
df_clean["work_interfere"] = df_clean["work_interfere"].fillna(work_interfere_mode)

# Check missing values after handling
print("\nMode used for self_employed:", self_employed_mode)
print("Mode used for work_interfere:", work_interfere_mode)

print("\nMissing values after handling:")
print(df_clean[["self_employed", "work_interfere", "Age"]].isnull().sum())

Missing values before handling:
self_employed      18
work_interfere    264
Age                 0
dtype: int64

Mode used for self_employed: No
Mode used for work_interfere: Sometimes

Missing values after handling:
self_employed     0
work_interfere    0
Age               0
dtype: int64


In [10]:
# Check all remaining missing values
#전체 데이터에 남아 있는 결측치가 없는지 최종 확

print("Total missing values in each column:")
print(df_clean.isnull().sum())

print("\nTotal number of missing values:")
print(df_clean.isnull().sum().sum())

Total missing values in each column:
Age                          0
Gender                       0
Country                      0
self_employed                0
family_history               0
treatment                    0
work_interfere               0
no_employees                 0
remote_work                  0
tech_company                 0
benefits                     0
care_options                 0
wellness_program             0
seek_help                    0
anonymity                    0
leave                        0
mental_health_consequence    0
phys_health_consequence      0
coworkers                    0
supervisor                   0
mental_health_interview      0
phys_health_interview        0
mental_vs_physical           0
obs_consequence              0
dtype: int64

Total number of missing values:
0


#Categroical value 처리
target variable인 treatment부터 숫자로 바꾸기 /treatment = Yes -> 1 , treatment = No -> 0

In [11]:
# Encode target variable: treatment

df_clean["treatment"] = df["treatment"].map({"Yes": 1, "No": 0})

print(df_clean["treatment"].value_counts())
print("Missing values in treatment:", df_clean["treatment"].isnull().sum())


treatment
1    637
0    622
Name: count, dtype: int64
Missing values in treatment: 0


#나머지 범주형 데이터 encoding하기
Dummy variables를 사용


In [12]:
# Step. Convert Country to Continent before dummy encoding

continent_map = {
    # North America
    "United States": "North America",
    "Canada": "North America",
    "Mexico": "North America",
    "Costa Rica": "North America",
    "Bahamas, The": "North America",

    # South America
    "Brazil": "South America",
    "Colombia": "South America",
    "Uruguay": "South America",

    # Europe
    "Austria": "Europe",
    "Belgium": "Europe",
    "Bosnia and Herzegovina": "Europe",
    "Bulgaria": "Europe",
    "Croatia": "Europe",
    "Czech Republic": "Europe",
    "Denmark": "Europe",
    "Finland": "Europe",
    "France": "Europe",
    "Georgia": "Europe",
    "Germany": "Europe",
    "Greece": "Europe",
    "Hungary": "Europe",
    "Ireland": "Europe",
    "Italy": "Europe",
    "Latvia": "Europe",
    "Moldova": "Europe",
    "Netherlands": "Europe",
    "Norway": "Europe",
    "Poland": "Europe",
    "Portugal": "Europe",
    "Romania": "Europe",
    "Russia": "Europe",
    "Slovenia": "Europe",
    "Spain": "Europe",
    "Sweden": "Europe",
    "Switzerland": "Europe",
    "United Kingdom": "Europe",

    # Asia
    "China": "Asia",
    "India": "Asia",
    "Israel": "Asia",
    "Japan": "Asia",
    "Philippines": "Asia",
    "Singapore": "Asia",
    "Thailand": "Asia",

    # Oceania
    "Australia": "Oceania",
    "New Zealand": "Oceania",

    # Africa
    "Nigeria": "Africa",
    "South Africa": "Africa",
    "Zimbabwe": "Africa"
}

# Create new Continent column from Country
df_clean["Continent"] = df_clean["Country"].map(continent_map)

# If there are countries not included in the map, classify them as Other
df_clean["Continent"] = df_clean["Continent"].fillna("Other")

# Drop original Country column
df_clean = df_clean.drop(columns=["Country"])

# Check result
print("Continent values:")
print(df_clean["Continent"].value_counts())

print("\nRemaining columns:")
print(df_clean.columns)

Continent values:
Continent
North America    828
Europe           362
Oceania           29
Asia              23
South America      9
Africa             8
Name: count, dtype: int64

Remaining columns:
Index(['Age', 'Gender', 'self_employed', 'family_history', 'treatment',
       'work_interfere', 'no_employees', 'remote_work', 'tech_company',
       'benefits', 'care_options', 'wellness_program', 'seek_help',
       'anonymity', 'leave', 'mental_health_consequence',
       'phys_health_consequence', 'coworkers', 'supervisor',
       'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence', 'Continent'],
      dtype='object')


In [13]:
# Encode categorical features using dummy variables

# Check categorical columns
categorical_columns = df_clean.select_dtypes(include=["object"]).columns

print("Categorical columns:")
print(categorical_columns)

df_encoded = pd.get_dummies(df_clean, columns=categorical_columns)

# Convert True/False dummy values to 1/0
df_encoded = df_encoded.astype(int)

print("\nBefore encoding:", df_clean.shape)
print("After encoding:", df_encoded.shape)

df_encoded.head()

Categorical columns:
Index(['Gender', 'self_employed', 'family_history', 'work_interfere',
       'no_employees', 'remote_work', 'tech_company', 'benefits',
       'care_options', 'wellness_program', 'seek_help', 'anonymity', 'leave',
       'mental_health_consequence', 'phys_health_consequence', 'coworkers',
       'supervisor', 'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence', 'Continent'],
      dtype='object')

Before encoding: (1259, 24)
After encoding: (1259, 72)


,Age,treatment,Gender_Female,Gender_Male,Gender_Other,self_employed_No,self_employed_Yes,family_history_No,family_history_Yes,work_interfere_Never,...,mental_vs_physical_No,mental_vs_physical_Yes,obs_consequence_No,obs_consequence_Yes,Continent_Africa,Continent_Asia,Continent_Europe,Continent_North America,Continent_Oceania,Continent_South America
0,37,1,1,0,0,1,0,1,0,0,...,0,1,1,0,0,0,0,1,0,0
1,44,0,0,1,0,1,0,1,0,0,...,0,0,1,0,0,0,0,1,0,0
2,32,0,0,1,0,1,0,1,0,0,...,1,0,1,0,0,0,0,1,0,0
3,31,1,0,1,0,1,0,0,1,0,...,1,0,0,1,0,0,1,0,0,0
4,31,0,0,1,0,1,0,1,0,1,...,0,0,1,0,0,0,0,1,0,0


#Feature Scaling
*   target인 treatment는 scaling하지 않고
*   나머지 Feature만 scaling 진행
*   Min-Max Scaling 사용



In [14]:
#Feature Scaling using Min-Max Scaling

from sklearn.preprocessing import MinMaxScaler

# Make a copy of encoded dataset
df_scaled = df_encoded.copy()

# Separate target column
target_column = "treatment"

# Select feature columns except target
feature_columns = df_scaled.columns.drop(target_column)

# Apply Min-Max Scaling to feature columns
scaler = MinMaxScaler()
df_scaled[feature_columns] = scaler.fit_transform(df_scaled[feature_columns])

# Check result
print("Before scaling:")
print(df_encoded[["Age"]].head())

print("\nAfter scaling:")
print(df_scaled[["Age"]].head())

print("\nFinal scaled dataset shape:", df_scaled.shape)
df_scaled.head()

Before scaling:
   Age
0   37
1   44
2   32
3   31
4   31

After scaling:
        Age
0  0.351852
1  0.481481
2  0.259259
3  0.240741
4  0.240741

Final scaled dataset shape: (1259, 72)


,Age,treatment,Gender_Female,Gender_Male,Gender_Other,self_employed_No,self_employed_Yes,family_history_No,family_history_Yes,work_interfere_Never,...,mental_vs_physical_No,mental_vs_physical_Yes,obs_consequence_No,obs_consequence_Yes,Continent_Africa,Continent_Asia,Continent_Europe,Continent_North America,Continent_Oceania,Continent_South America
0,0.351852,1,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.481481,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.259259,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.240741,1,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.240741,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [15]:
# Final check and save the preprocessed dataset

print("Final dataset shape:", df_scaled.shape)

print("\nTotal missing values:")
print(df_scaled.isnull().sum().sum())

print("\nData types:")
print(df_scaled.dtypes.value_counts())

print("\nTreatment values:")
print(df_scaled["treatment"].value_counts())

# Save the final preprocessed dataset
df_scaled.to_csv("preprocessed_survey.csv", index=False)

print("\nSaved file: preprocessed_survey.csv")

Final dataset shape: (1259, 72)

Total missing values:
0

Data types:
float64    71
int64       1
Name: count, dtype: int64

Treatment values:
treatment
1    637
0    622
Name: count, dtype: int64

Saved file: preprocessed_survey.csv
